# Imports


In [ ]:
import duckdb
import polars as pl
from pathlib import Path

# Data Path


In [ ]:
data_folder = Path("../data")

# Connect to DuckDB


In [ ]:
con = duckdb.connect()

## Query Parquet File


In [ ]:
sales = con.execute(f"""
    select * from read_parquet('{data_folder}/sales_cleaned.parquet')
    limit 5
""").fetchdf()

sales

# SQL Revenue by Category


In [ ]:
category_sales = con.execute(f"""
    
    select category,
    count(*) as transactions,
    sum(revenue) as total_revenue,
    round(avg(revenue)) as avg_order_value
    from read_parquet('{data_folder}/sales_cleaned.parquet')
    group by category
    order by total_revenue desc                         

    """).fetchdf()

print(category_sales)

# SQL Monthly Revenue


In [ ]:
monthly_sale_sql = con.execute(f"""

    select strftime('%Y-%m',purchase_date) as month,
    count(*) as transactions,
    sum(revenue) as total_revenue,
    from read_parquet('{data_folder}/sales_cleaned.parquet')
    group by month
    order by month

    """).fetchdf()

print(monthly_sale_sql)

# Customer Segmentation SQL Table


In [ ]:
customer_metrics_sql = con.execute(f"""

    select customer_id,
    count(*) as total_orders,
    sum(revenue) as total_spent,
    avg(revenue) as avg_order_value,
    min(purchase_date) as first_purchase,
    max(purchase_date) as last_purchase,
    from read_parquet('{data_folder}/sales_cleaned.parquet')
    group by customer_id

    """).fetchdf()

customer_metrics_sql.head()

# City Performance SQL Table


In [ ]:
city_performance_sql = con.execute(f"""

    select city,
    count(*) as transactions,
    sum(revenue) as total_revenue,
    round(sum(revenue) * 100.0 /sum(sum(revenue)) over(),2) as revenue_share_pct
    from read_parquet('{data_folder}/sales_cleaned.parquet')
    group by city
    order by total_revenue desc                        

    """).fetchdf()

print(city_performance_sql)

# Festival & Monthly Analysis


In [ ]:
monthly_festival_sql = con.execute(f"""

    select strftime('%Y-%m',purchase_date) as month,
    festival,
    count(*) as transactions,
    sum(revenue) as total_revenue,
    from read_parquet('{data_folder}/sales_cleaned.parquet')
    group by month,festival
    order by month

    """).fetchdf()

print(monthly_festival_sql)

# Product performance table


In [ ]:

product_perf_sql = con.execute(f"""

    select product_name,
    category,
    count(*) as transactions,
    sum(revenue) as total_revenue,
    round(avg(revenue)) as avg_order_value
    from read_parquet('{data_folder}/sales_cleaned.parquet')
    group by product_name,category
    order by total_revenue desc                         

    """).fetchdf()

print(product_perf_sql)

# Save BI-ready tables to Parquet


In [ ]:
customer_metrics_sql.to_parquet(
    data_folder / "customer_metrics_duckdb.parquet")
city_performance_sql.to_parquet(
    data_folder / "city_performance_duckdb.parquet")
monthly_festival_sql.to_parquet(
    data_folder / "monthly_festival_duckdb.parquet")
product_perf_sql.to_parquet(data_folder / "products_perf_duckdb.parquet")

## Verify the Parquet files


In [ ]:
print("Customer metrics rows:", len(customer_metrics_sql))
print("City performance rows:", len(city_performance_sql))
print("Monthly festival rows:", len(monthly_festival_sql))
print("Product performance rows:", len(product_perf_sql))